# cAST-Scope — comparaison des 3 retrievers (50 tâches)

Notebook léger, dédié uniquement à comparer `bm25` / `codesage` / `agentic` sur RepoEval — pas le run complet à 300 tâches, pas CodeLlama, juste ça. Voir `colab_benchmark.ipynb` pour le run complet.

GPU nécessaire (CodeSage + StarCoder2-7B) — T4 suffit pour ce volume.

## 0. Vérifier le GPU

In [ ]:
!nvidia-smi

## 1. Cloner (ou mettre à jour) le dépôt + installer les dépendances

In [ ]:
import os

if os.path.isdir('/content/cAST-state'):
    %cd /content/cAST-state
    !git pull
else:
    !git clone --depth 1 https://github.com/Robertkiza0/cAST-state.git /content/cAST-state
    %cd /content/cAST-state

SENTINEL = '/content/.cast_state_installed'
if not os.path.exists(SENTINEL):
    !pip install -q -r requirements.txt
    !pip install -q transformers accelerate editdistance
    open(SENTINEL, 'w').close()
    print()
    print('=' * 70)
    print('INSTALLATION TERMINEE. ETAPE OBLIGATOIRE MAINTENANT :')
    print('  Menu Runtime/Exécution > Restart session / Redémarrer la session')
    print('Puis RELANCEZ CETTE MEME CELLULE UNE FOIS, et continuez.')
    print('=' * 70)
else:
    print()
    !echo "=== Commit actif : $(git rev-parse --short HEAD) — $(git log -1 --format=%s) ==="


## 1bis. Token Hugging Face

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN chargé depuis les secrets Colab.")
except Exception:
    from huggingface_hub import login
    login()
    print("Connecté via huggingface_hub.login().")


## 2. Télécharger les données RepoEval (tâches + 8 dépôts réels)

Idempotent : si déjà présent, ne retélécharge rien.

In [ ]:
import os
import zipfile

if os.path.isdir('data/repos_source') and os.listdir('data/repos_source'):
    print('RepoEval déjà présent, rien à faire.')
else:
    !rm -rf codet_src
    !git clone --no-checkout --depth 1 https://github.com/microsoft/CodeT.git codet_src
    %cd codet_src
    !git sparse-checkout init --cone
    !git sparse-checkout set RepoCoder
    !git checkout main
    %cd ..

    with zipfile.ZipFile('codet_src/RepoCoder/datasets/datasets.zip') as z:
        z.extractall('datasets rapo')
    with zipfile.ZipFile('codet_src/RepoCoder/repositories/line_and_api_level.zip') as z:
        z.extractall('data/repos_source')

    print('RepoEval : dataset et dépôts extraits.')


## 3. Sanity check (générateur factice, valide que CodeSage charge et que le reranking ne plante pas)

In [ ]:
%run -i /content/cAST-state/run_benchmark.py --dataset repoeval --n-tasks 10 \
    --generator stub --retriever codesage


## 4. Comparaison réelle — 50 tâches, StarCoder2-7B, un run par retriever

In [ ]:
for retriever in ['bm25', 'codesage', 'agentic']:
    print(f'\n{"="*20} RETRIEVER: {retriever} {"="*20}')
    %run -i /content/cAST-state/run_benchmark.py --dataset repoeval --n-tasks 50 \
        --generator hf --model-name bigcode/starcoder2-7b --retriever $retriever


## 5. Significativité (McNemar) entre les 3 retrievers

Comme pour les baselines de chunking : ne pas conclure sur les moyennes brutes seules. Compare les 3 derniers fichiers de résultats (un par retriever, dans l'ordre où ils ont été produits ci-dessus) deux à deux.

In [ ]:
!pip install -q scipy

import glob

runs = sorted(glob.glob('/content/cAST-state/results/run_*.jsonl'))[-3:]
labels = ['bm25', 'codesage', 'agentic']
for label, run_path in zip(labels, runs):
    print(f'\n### {label}: {run_path}')
    %run -i /content/cAST-state/experiments/mcnemar_significance.py $run_path


## Télécharger les résultats (le runtime Colab est éphémère)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/cast_state_retriever_results', 'zip', '/content/cAST-state/results')
files.download('/content/cast_state_retriever_results.zip')


## Notes

- `codesage`/`agentic` ont besoin de `torch` (installé par défaut sur Colab) — jamais testés en local par Claude (pas de torch sur cette machine), donc surveillez la cellule 3 en premier.
- L'`agentic` reranker rappelle un signal de scoring déjà invalidé sur un autre axe de ce projet (voir mémoire) — à garder en tête en interprétant les résultats, pas une raison de s'arrêter.
- Toutes les cellules d'installation sont idempotentes — sûres à relancer après un redémarrage.